# Прогнозирование временных рядов

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Прогнозирование временных рядов».

## Что делает метод

Временной ряд — это величина, измеренная последовательно во времени: цена, температура, заболеваемость, показатель процесса. Прогнозирование отвечает на вопрос «что будет дальше», опираясь на прошлую динамику. Алгоритм улавливает устойчивые закономерности — тренд, повторяющиеся сезонные колебания — и продолжает их в будущее.

В этом блокноте мы построим прогноз методом признаков-лагов с градиентным бустингом, корректно проверим его на отложенном будущем и оценим интервал неопределённости. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что вы каждое утро измеряете температуру воздуха за окном и хотите угадать, какой она будет через месяц. У вас есть два подсказки: во-первых, общая тенденция (лето теплее зимы — это тренд, то есть долгосрочное направление изменений); во-вторых, повторяющийся годовой цикл (сезонность — регулярно повторяющийся узор колебаний). Зная вчерашнюю температуру, температуру неделю назад и то, какое сейчас время года, вы можете сделать обоснованный прогноз.

Именно это делает метод: он собирает «воспоминания» ряда — значения за прошлые периоды (лаги, то есть прошлые значения со сдвигом во времени) и скользящие средние (среднее за последние N дней) — и превращает задачу предсказания будущего в обычную задачу регрессии. Модель обучается на этих признаках и выдаёт не только точечный прогноз, но и интервал неопределённости — диапазон, внутри которого, скорее всего, окажется будущее значение.

**Горизонт прогноза** — насколько далеко в будущее мы предсказываем. Чем дальше горизонт, тем шире честный интервал неопределённости.

## 1. Установка библиотек

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Сформируем синтетический суточный ряд за несколько лет: устойчивый тренд, годовая сезонность и случайный шум. Такой ряд воспроизводит типичную структуру научных временных данных (заболеваемость, метеопоказатель, нагрузка процесса) и не требует загрузки внешних файлов.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n_days = 365 * 4
dates = pd.date_range("2021-01-01", periods=n_days, freq="D")
t = np.arange(n_days)

trend = 50 + 0.03 * t
seasonal = 12 * np.sin(2 * np.pi * t / 365.25)
noise = rng.normal(0, 3, n_days)
value = trend + seasonal + noise

series = pd.Series(value, index=dates, name="показатель")
print(f"Длина ряда: {len(series)} наблюдений")
print(f"Период: {series.index.min().date()} - {series.index.max().date()}")
series.head()

## 4. Применение метода

### Шаг 1. Инженерия признаков: превращаем ряд в таблицу

Алгоритм градиентного бустинга (метод, строящий ансамбль решающих деревьев, где каждое следующее дерево исправляет ошибки предыдущего) не умеет работать с временным рядом напрямую. Мы переводим его в таблицу: для каждого дня собираем признаки из **прошлого** — лаги (значения за 1, 7, 14 и 365 дней до текущего), скользящие средние за 7 и 30 дней, а также календарные характеристики (день года, день недели).

**Важное правило:** все признаки должны быть известны до прогнозируемого момента. Использование информации из будущего — так называемая утечка данных — приведёт к иллюзорно хорошим результатам, которые не воспроизведутся на реальных данных.

Далее ряд делится на обучающую часть и проверочную (последние 120 дней). Граница проходит строго по времени: нельзя перемешивать данные и брать проверочные точки из середины ряда.

In [ ]:
def build_features(s):
    """Формирует таблицу признаков из прошлого ряда."""
    df = pd.DataFrame({"y": s})
    for lag in (1, 7, 14, 365):
        df[f"лаг_{lag}"] = s.shift(lag)
    df["скользящее_7"] = s.shift(1).rolling(7).mean()
    df["скользящее_30"] = s.shift(1).rolling(30).mean()
    df["день_года"] = s.index.dayofyear
    df["день_недели"] = s.index.dayofweek
    return df.dropna()


features = build_features(series)
X = features.drop(columns=["y"])
y = features["y"]

# Разделение по времени: тест - последние 120 дней, без перемешивания.
horizon = 120
X_train, X_test = X.iloc[:-horizon], X.iloc[-horizon:]
y_train, y_test = y.iloc[:-horizon], y.iloc[-horizon:]
print(f"Обучение: {len(X_train)} дней, проверка: {len(X_test)} дней")

### Шаг 2. Обучение модели и построение интервала неопределённости

Обучаем три модели:
- **Центральная** — предсказывает ожидаемое значение.
- **Нижняя (квантиль 5 %)** — предсказывает нижнюю границу: с вероятностью 95 % истинное значение окажется выше.
- **Верхняя (квантиль 95 %)** — предсказывает верхнюю границу.

Квантиль — это уровень вероятности: квантиль 5 % означает «5 % наблюдений оказались ниже этой отметки». Полоса между квантилями 5 % и 95 % называется 90-процентным интервалом прогноза.

Для сравнения строим **сезонный наивный прогноз** — простая, но сильная базовая линия: берём значение ровно год назад. Если наша сложная модель не превосходит этот простой прогноз по точности, смысла в ней нет.

В конце ячейки выводятся значения **MAE** (средняя абсолютная ошибка) — среднее по всем дням отклонение прогноза от факта в единицах ряда.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

# Центральный прогноз и две квантильные модели для интервала неопределённости.
model = HistGradientBoostingRegressor(learning_rate=0.1, max_iter=300,
                                      random_state=42)
model.fit(X_train, y_train)
y_pred = pd.Series(model.predict(X_test), index=y_test.index)

lower_model = HistGradientBoostingRegressor(loss="quantile", quantile=0.05,
                                            max_iter=300, random_state=42)
upper_model = HistGradientBoostingRegressor(loss="quantile", quantile=0.95,
                                            max_iter=300, random_state=42)
lower_model.fit(X_train, y_train)
upper_model.fit(X_train, y_train)
y_low = pd.Series(lower_model.predict(X_test), index=y_test.index)
y_high = pd.Series(upper_model.predict(X_test), index=y_test.index)

# Базовый прогноз для сравнения: сезонный наивный (значение год назад).
y_naive = X_test["лаг_365"]

print(f"MAE модели:               {mean_absolute_error(y_test, y_pred):.3f}")
print(f"MAE сезонного наивного:    {mean_absolute_error(y_test, y_naive):.3f}")

### Шаг 3. Визуализация прогноза

Строим график: история ряда (серая линия), фактические значения проверочного периода (синяя линия), прогноз модели (оранжевая линия) и интервал неопределённости (закрашенная полоса). Это «честный» тест: модель никогда не видела эти 120 дней в ходе обучения.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(13, 5.4))

history = y_train.iloc[-180:]
ax.plot(history.index, history.values, color=VIZ["series"][3],
        label="История (обучение)")
ax.plot(y_test.index, y_test.values, color=VIZ["series"][0],
        label="Факт (проверка)")
ax.plot(y_pred.index, y_pred.values, color=VIZ["series"][2],
        label="Прогноз модели")
ax.fill_between(y_test.index, y_low.values, y_high.values,
                color=VIZ["series"][2], alpha=0.18,
                label="Интервал 5-95 %")
ax.set_title("Прогноз временного ряда на отложенном периоде")
ax.set_xlabel("Дата")
ax.set_ylabel("Показатель")
ax.legend(loc="upper left")
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

**Как читать этот график.**
- Серая линия — обучающая история: модель обучалась на этих данных.
- Синяя линия — фактические значения проверочного периода: их модель не видела.
- Оранжевая линия — прогноз: насколько она близка к синей, настолько модель точна.
- Закрашенная полоса — интервал неопределённости 5–95 %: примерно 9 из 10 реальных значений должны попасть внутрь. Если полоса слишком узкая и многие точки выходят за её пределы, интервал недооценивает неопределённость.
- Систематическое смещение прогноза вверх или вниз (линия всегда выше или ниже факта) — признак того, что модель не уловила какую-то структуру ряда.

## 5. Интерпретация результата

- **MAE** — средняя абсолютная ошибка прогноза в единицах ряда. Сравните с базовым (сезонным наивным) прогнозом: если MAE модели не ниже базового, то более сложная модель не оправдана на этих данных.
- **График**: линия прогноза должна повторять тренд и сезонность факта; систематическое отклонение указывает на смещение.
- **Интервал неопределённости**: полоса между квантилями показывает разброс возможных значений. Точечный прогноз без интервала создаёт ложное ощущение точности. Чем дальше горизонт прогноза, тем шире честный интервал.
- Проверка проведена только на данных, идущих позже по времени; случайное перемешивание для временных рядов недопустимо.

Помните: метод не предскажет структурный сдвиг, не представленный в исторических данных.

## 6. Поэкспериментируйте сами

Ниже — ячейка для самостоятельных экспериментов. Измените один параметр, снова запустите ячейки раздела 4 и сравните результат.

**Что менять и что наблюдать:**
- Измените `horizon` (строка `horizon = 120` в разделе 4) на 30 или 200. Как меняется качество прогноза при более коротком и более длинном горизонте?
- Уберите из списка лагов `365` (годовой лаг). Что происходит с прогнозом: пропадёт ли сезонность?
- Увеличьте шум в данных: в разделе 3 измените `noise = rng.normal(0, 3, n_days)` на `rng.normal(0, 10, n_days)`. Как расширится интервал неопределённости?
- Уберите тренд (установите `trend = 50 + 0 * t`). Стала ли модель точнее без долгосрочного дрейфа?

In [ ]:
# --- Шаблон загрузки своих данных ---
# raw = pd.read_csv("my_series.csv")            # ваш файл
# date_column = "дата"                          # имя столбца с датой
# value_column = "значение"                     # имя столбца со значением
#
# raw[date_column] = pd.to_datetime(raw[date_column])
# series = raw.set_index(date_column)[value_column].sort_index().asfreq("D")
# series = series.interpolate()                 # заполнение редких пропусков
#
# print(f"Длина ряда: {len(series)} наблюдений")
# Далее повторите ячейки раздела 4.

## Краткие выводы

- Прогнозирование временного ряда сводится к регрессии на признаках из прошлого (лагах и скользящих средних) — без утечки информации из будущего.
- Тренд — долгосрочное направление изменений; сезонность — повторяющийся периодический узор. Оба компонента модель должна уметь воспроизвести.
- Горизонт прогноза — насколько далеко в будущее мы смотрим. Чем он больше, тем шире должен быть интервал неопределённости.
- Всегда сравнивайте сложную модель с простой базовой линией: если разница незначительна, простота предпочтительнее.
- Проверка только на хронологически более поздних данных — обязательное условие честной оценки качества прогноза.

## 7. Попробуйте на своих данных

Замените демонстрационный ряд своими данными. Подготовьте таблицу с двумя столбцами: дата и значение показателя.

1. Загрузите файл в Colab: панель слева, вкладка «Файлы», кнопка загрузки.
2. Снимите комментарии в ячейке ниже, укажите имена столбцов и при необходимости скорректируйте набор лагов под период сезонности своего ряда.
3. Последовательно выполните ячейки разделов 4 и 5.